# Notebook 10: Embedding Systems at Scale

## Why This Matters

Embeddings are the foundational primitive of modern ML systems—from recommendation engines to semantic search to LLMs. Pinterest's PinSage, Spotify's track embeddings, Airbnb's listing embeddings, and Instacart's product embeddings all power real-time personalization at massive scale. The engineering challenges of embedding systems—training embeddings over billion-node graphs, serving them with sub-millisecond ANN lookup from billion-item catalogs, managing embedding freshness, and handling cold start—appear in nearly every large-scale ML system design interview. Understanding how approximate nearest neighbor (ANN) search works and which index to choose (Faiss, ScaNN, HNSW) is expected knowledge.

## 1. What Are Embeddings and Why Do They Matter?

An embedding is a dense, low-dimensional representation of a high-dimensional or discrete entity (user, item, query, document). Embeddings capture semantic similarity—items with similar meaning or behavior are close in embedding space. This makes them the core representation for recommendation, search, and classification at scale.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

np.random.seed(42)

# Simulate item embeddings in 64-dimensional space
# Categories: tech, sports, cooking, travel — items within categories cluster together
n_per_category = 30
dim = 64

categories = {
    'Tech Products': np.random.normal([1]*dim, 0.3, (n_per_category, dim)),
    'Sports Items': np.random.normal([-1]*dim + [0]*(dim//2), 0.3, (n_per_category, dim)),
    'Cooking/Food': np.random.normal([0]*dim, 0.3, (n_per_category, dim)),
    'Travel/Hotels': np.random.normal([1, -1]*( dim//2), 0.3, (n_per_category, dim)),
}

# Make embeddings more separable
cluster_centers = {
    'Tech Products': np.concatenate([[3, 3], np.zeros(dim-2)]),
    'Sports Items': np.concatenate([[-3, 3], np.zeros(dim-2)]),
    'Cooking/Food': np.concatenate([[0, -3], np.zeros(dim-2)]),
    'Travel/Hotels': np.concatenate([[-3, -3], np.zeros(dim-2)]),
}

all_embeddings = []
all_labels = []
colors = ['#E74C3C', '#3498DB', '#27AE60', '#F39C12']

for (cat, center), color in zip(cluster_centers.items(), colors):
    embs = np.random.normal(0, 0.5, (n_per_category, dim))
    embs += center
    all_embeddings.append(embs)
    all_labels.extend([cat] * n_per_category)

X = np.vstack(all_embeddings)

# Reduce to 2D for visualization
tsne = TSNE(n_components=2, random_state=42, perplexity=20)
X_2d = tsne.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
for (cat, _), color in zip(cluster_centers.items(), colors):
    mask = [l == cat for l in all_labels]
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, s=50, alpha=0.7, label=cat)

ax.set_title('Item Embeddings Visualized (t-SNE)\nSemantically similar items cluster together',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlabel('t-SNE dimension 1')
ax.set_ylabel('t-SNE dimension 2')

# Cosine similarity between embeddings
ax2 = axes[1]
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

# Sample 8 items from different categories for a similarity matrix
sample_idx = [0, 5, 30, 35, 60, 65, 90, 95]  # 2 from each category
sample_labels = [all_labels[i] for i in sample_idx]
sample_names = [f"{l.split('/')[0][:6]}-{i%30}" for i, l in zip(sample_idx, sample_labels)]
sample_embs = X[sample_idx]

sim_matrix = np.zeros((len(sample_idx), len(sample_idx)))
for i in range(len(sample_idx)):
    for j in range(len(sample_idx)):
        sim_matrix[i, j] = cosine_similarity(sample_embs[i], sample_embs[j])

im = ax2.imshow(sim_matrix, cmap='RdYlGn', vmin=-0.5, vmax=1.0)
ax2.set_xticks(range(len(sample_names)))
ax2.set_yticks(range(len(sample_names)))
ax2.set_xticklabels(sample_names, rotation=45, ha='right', fontsize=8)
ax2.set_yticklabels(sample_names, fontsize=8)
ax2.set_title('Cosine Similarity Matrix\n(Same-category items = high similarity)', fontsize=11, fontweight='bold')

for i in range(len(sample_idx)):
    for j in range(len(sample_idx)):
        ax2.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center',
                 fontsize=8, color='white' if sim_matrix[i,j] < 0.3 else '#333')

plt.colorbar(im, ax=ax2)
plt.tight_layout()
plt.savefig('embedding_visualization.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Embedding dimension: {dim}")
print(f"Memory for 1M items (float32): {1_000_000 * dim * 4 / 1e9:.2f} GB")
print(f"Memory for 1B items (float32): {1_000_000_000 * dim * 4 / 1e9:.1f} GB")
print("→ 1B item embeddings at dim=256 = 1TB → requires distributed storage or ANN with compression")

## 2. Approximate Nearest Neighbor (ANN) Search

Exact nearest neighbor search in billion-item catalogs is O(n × d) per query—too slow for real-time serving. ANN search trades a small accuracy loss for orders-of-magnitude speed improvement. Understanding the major ANN algorithms and their tradeoffs is critical for retrieval system design.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from time import perf_counter

def exact_nearest_neighbors(query, database, k=10):
    """Brute force: O(n*d). Correct but slow."""
    # Cosine similarity via dot product (assuming unit vectors)
    scores = database @ query
    top_k_idx = np.argsort(scores)[-k:][::-1]
    return top_k_idx, scores[top_k_idx]

def inverted_file_index_ann(query, database, centroids, assignments, k=10, n_probe=5):
    """
    IVF (Inverted File Index): partition database into clusters,
    search only the nearest n_probe clusters.
    This is the core of Faiss's IVF method.
    """
    # Find nearest clusters to query
    centroid_scores = centroids @ query
    nearest_clusters = np.argsort(centroid_scores)[-n_probe:]
    
    # Search within those clusters
    candidate_idx = []
    for cluster in nearest_clusters:
        candidate_idx.extend(np.where(assignments == cluster)[0].tolist())
    
    if not candidate_idx:
        return np.array([]), np.array([])
    
    candidates = database[candidate_idx]
    scores = candidates @ query
    local_top_k = np.argsort(scores)[-k:][::-1]
    return np.array(candidate_idx)[local_top_k], scores[local_top_k]

# Benchmark
np.random.seed(42)
dim = 128
n_items = 100_000  # 100K items for demo
k = 10

# Unit normalized embeddings
database = np.random.normal(0, 1, (n_items, dim))
database = database / np.linalg.norm(database, axis=1, keepdims=True)

# Build IVF index
n_clusters = 100
from sklearn.cluster import MiniBatchKMeans
kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, n_init=3)
assignments = kmeans.fit_predict(database)
centroids = kmeans.cluster_centers_
centroids = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)

# Benchmark queries
n_queries = 100
queries = np.random.normal(0, 1, (n_queries, dim))
queries = queries / np.linalg.norm(queries, axis=1, keepdims=True)

# Exact search
t0 = perf_counter()
exact_results = [exact_nearest_neighbors(q, database, k=k) for q in queries]
exact_time = (perf_counter() - t0) / n_queries * 1000

# ANN search (n_probe=5)
t0 = perf_counter()
ann_results_5 = [inverted_file_index_ann(q, database, centroids, assignments, k=k, n_probe=5) for q in queries]
ann_time_5 = (perf_counter() - t0) / n_queries * 1000

# ANN search (n_probe=20)
t0 = perf_counter()
ann_results_20 = [inverted_file_index_ann(q, database, centroids, assignments, k=k, n_probe=20) for q in queries]
ann_time_20 = (perf_counter() - t0) / n_queries * 1000

# Compute recall
def recall_at_k(exact, approx, k):
    if len(approx[0]) == 0: return 0
    true_set = set(exact[0][:k])
    approx_set = set(approx[0][:k])
    return len(true_set & approx_set) / k

recall_5 = np.mean([recall_at_k(e, a, k) for e, a in zip(exact_results, ann_results_5)])
recall_20 = np.mean([recall_at_k(e, a, k) for e, a in zip(exact_results, ann_results_20)])

print(f"ANN Benchmark on {n_items:,} items (dim={dim}, k={k}):")
print(f"{'Method':25s} {'Avg Latency':>15} {'Recall@{k}':>12} {'Speedup':>10}")
print('-' * 65)
print(f"{'Exact (brute force)':25s} {exact_time:>12.2f}ms {1.0:>12.2f} {'1.0x':>10}")
print(f"{'IVF ANN (n_probe=5)':25s} {ann_time_5:>12.2f}ms {recall_5:>12.2f} {exact_time/ann_time_5:>9.1f}x")
print(f"{'IVF ANN (n_probe=20)':25s} {ann_time_20:>12.2f}ms {recall_20:>12.2f} {exact_time/ann_time_20:>9.1f}x")
print("\nThe recall-speed tradeoff: more probes = higher recall, slower search")

## 3. ANN Index Comparison

Different ANN algorithms optimize for different tradeoffs. The choice between HNSW, IVF, ScaNN, and LSH depends on your dataset size, memory constraints, update frequency, and recall requirements.

In [ ]:
import pandas as pd

ann_comparison = pd.DataFrame([
    {
        'Algorithm': 'HNSW (Hierarchical NSW)',
        'Library': 'hnswlib, Faiss',
        'Recall@10': '0.97-0.99',
        'Query Speed': 'Fastest (sub-ms at 1B)',
        'Build Time': 'Slow (hours for 1B)',
        'Memory': 'High (graph structure)',
        'Supports Updates': 'Yes (append)',
        'Best For': 'High recall requirements, static index, fits in RAM',
    },
    {
        'Algorithm': 'IVF + PQ (Faiss)',
        'Library': 'Faiss',
        'Recall@10': '0.85-0.95',
        'Query Speed': 'Fast (1-5ms)',
        'Build Time': 'Medium',
        'Memory': 'Low (quantization)',
        'Supports Updates': 'Batch rebuild required',
        'Best For': 'Memory-constrained, billion-scale, lower recall OK',
    },
    {
        'Algorithm': 'ScaNN (Google)',
        'Library': 'google-research/scann',
        'Recall@10': '0.95-0.98',
        'Query Speed': 'Very fast (SIMD optimized)',
        'Build Time': 'Medium',
        'Memory': 'Medium',
        'Supports Updates': 'Batch rebuild',
        'Best For': 'Maximum throughput on CPU, used by Google',
    },
    {
        'Algorithm': 'LSH (Locality Sensitive Hashing)',
        'Library': 'Custom / Pinecone',
        'Recall@10': '0.70-0.85',
        'Query Speed': 'Fast',
        'Build Time': 'Fast',
        'Memory': 'Low',
        'Supports Updates': 'Yes (easy)',
        'Best For': 'Dynamic indexes, approximate recall OK, fast updates',
    },
    {
        'Algorithm': 'Disk ANN (Microsoft)',
        'Library': 'Microsoft DiskANN',
        'Recall@10': '0.93-0.96',
        'Query Speed': 'Medium (SSD I/O bound)',
        'Build Time': 'Medium',
        'Memory': 'Very low (disk-based)',
        'Supports Updates': 'Yes',
        'Best For': 'Index too large for RAM, disk-based serving',
    },
    {
        'Algorithm': 'Pinecone / Weaviate',
        'Library': 'Managed service',
        'Recall@10': '0.90-0.97',
        'Query Speed': '5-20ms (network)',
        'Build Time': 'Managed',
        'Memory': 'Managed',
        'Supports Updates': 'Yes (fully managed)',
        'Best For': 'Startup/prototype, willing to pay for managed',
    },
])

print("ANN Algorithm Comparison:")
for _, row in ann_comparison.iterrows():
    print(f"\n[{row['Algorithm']}]")
    for col in ['Library', 'Recall@10', 'Query Speed', 'Memory', 'Supports Updates', 'Best For']:
        print(f"  {col}: {row[col]}")

print("\nInterview rule of thumb:")
print("  High recall + static index?      → HNSW")
print("  Billion scale + memory limited?  → Faiss IVF-PQ")
print("  Maximum CPU throughput?          → ScaNN (Google's choice)")
print("  Dynamic index with easy updates? → LSH or managed (Pinecone)")
print("  Index too big for RAM?           → DiskANN")

## 4. Two-Tower Model Architecture

The two-tower (dual encoder) model is the dominant architecture for generating embeddings for recommendation and search. It learns separate encoders for queries/users and items, enabling efficient pre-computation of item embeddings while keeping user encoders fresh.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, axes = plt.subplots(1, 2, figsize=(16, 9))

def box(ax, x, y, w, h, text, color, fontsize=9):
    r = FancyBboxPatch((x,y), w, h, boxstyle="round,pad=0.1",
                       facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.9)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color='white', multialignment='center')

def arr(ax, x1, y1, x2, y2, label='', color='#555'):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8))
    if label:
        ax.text((x1+x2)/2+0.1, (y1+y2)/2+0.1, label, fontsize=8, color=color)

# Architecture diagram
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
ax.set_title('Two-Tower Architecture', fontsize=12, fontweight='bold')

# User tower
box(ax, 0.3, 7.5, 3.5, 1.2, 'User Features\n(history, demographics, context)', '#7F8C8D')
box(ax, 0.3, 5.5, 3.5, 1.5, 'User Encoder\n(DNN layers)', '#2980B9')
box(ax, 0.8, 3.8, 2.5, 1.2, 'User Embedding\n(256-dim)', '#3498DB')

# Item tower
box(ax, 6, 7.5, 3.5, 1.2, 'Item Features\n(content, metadata, stats)', '#7F8C8D')
box(ax, 6, 5.5, 3.5, 1.5, 'Item Encoder\n(DNN layers)', '#8E44AD')
box(ax, 6.5, 3.8, 2.5, 1.2, 'Item Embedding\n(256-dim)', '#9B59B6')

# Similarity
box(ax, 3.5, 2.0, 3, 1.2, 'Dot Product\nSimilarity Score', '#27AE60')
box(ax, 3.5, 0.3, 3, 1.2, 'Training Loss\n(Softmax over batch)', '#E74C3C')

# Training pairs
ax.text(5, 9, 'During Training: (user, item, label) pairs', ha='center', fontsize=9, style='italic', color='#555')

# Arrows
arr(ax, 2.05, 7.5, 2.05, 7.0)
arr(ax, 2.05, 5.5, 2.05, 5.0)
arr(ax, 7.75, 7.5, 7.75, 7.0)
arr(ax, 7.75, 5.5, 7.75, 5.0)
arr(ax, 2.05, 3.8, 4.0, 3.2)
arr(ax, 7.75, 3.8, 6.0, 3.2)
arr(ax, 5, 2.0, 5, 1.5)

# Serving annotations
ax.text(2.05, 3.2, '↑ Computed\nper request', ha='center', fontsize=7.5, color='#2980B9', style='italic')
ax.text(7.75, 3.2, '↑ Pre-computed\noffline', ha='center', fontsize=7.5, color='#8E44AD', style='italic')

# Serving flow
ax2 = axes[1]
ax2.set_xlim(0, 10); ax2.set_ylim(0, 10); ax2.axis('off')
ax2.set_title('Two-Tower Serving: Offline + Online', fontsize=12, fontweight='bold')

# Offline path
box(ax2, 0.3, 8.5, 4, 1, 'All Items (1B+)', '#7F8C8D')
box(ax2, 0.3, 7, 4, 1, 'Item Encoder (batch)', '#8E44AD')
box(ax2, 0.3, 5.5, 4, 1, '1B Item Embeddings', '#9B59B6')
box(ax2, 0.3, 4, 4, 1, 'ANN Index (Faiss/ScaNN)', '#E74C3C')

arr(ax2, 2.3, 8.5, 2.3, 8.0)
arr(ax2, 2.3, 7.0, 2.3, 6.5)
arr(ax2, 2.3, 5.5, 2.3, 5.0)

ax2.text(2.3, 3.5, 'OFFLINE PATH\n(updated daily/hourly)', ha='center', fontsize=8.5,
         color='#8E44AD', fontweight='bold')

# Online path
box(ax2, 5.5, 8.5, 4, 1, 'User Request\n(user_id + context)', '#7F8C8D')
box(ax2, 5.5, 7, 4, 1, 'User Encoder (real-time)', '#2980B9')
box(ax2, 5.5, 5.5, 4, 1, 'User Embedding (256-dim)', '#3498DB')
box(ax2, 5.5, 4, 4, 1, 'ANN Lookup (top-1000)', '#27AE60')
box(ax2, 5.5, 2.5, 4, 1, 'Candidate Set → Ranker', '#E67E22')

arr(ax2, 7.5, 8.5, 7.5, 8.0)
arr(ax2, 7.5, 7.0, 7.5, 6.5)
arr(ax2, 7.5, 5.5, 7.5, 5.0)
arr(ax2, 7.5, 4.0, 7.5, 3.5)

ax2.text(7.5, 2.0, 'ONLINE PATH (<50ms)', ha='center', fontsize=8.5,
         color='#2980B9', fontweight='bold')

# Connect ANN index to lookup
arr(ax2, 4.3, 4.5, 5.5, 4.5, '', '#E74C3C')
ax2.text(4.9, 4.7, 'index\nlookup', ha='center', fontsize=7.5, color='#E74C3C')

plt.tight_layout()
plt.savefig('two_tower_architecture.png', dpi=120, bbox_inches='tight')
plt.show()

print("Key insight: Item embeddings are pre-computed (offline), user embeddings are computed at request time.")
print("This makes the online path fast: one user encoder forward pass + ANN lookup.")
print("\nUsed at: YouTube (two-tower retrieval), Airbnb listings, Pinterest PinSage, Spotify")

## 5. Embedding Freshness and Cold Start

Embeddings become stale as item/user behavior changes. New items (cold start) have no embedding until they appear in training data. Managing freshness and cold start are key engineering challenges.

In [ ]:
strategies = {
    'Item Embedding Freshness': {
        'Problem': 'Item embeddings trained on historical interactions become stale',
        'Approaches': [
            ('Periodic full retrain', 'Weekly/daily. Simple, consistent. Best for slowly evolving catalogs.'),
            ('Online embedding updates', 'Fine-tune embeddings with new interactions in near-real-time. Complex.'),
            ('Content-based fallback', 'For items without interaction history, use content encoder (text, images).'),
            ('Hybrid scoring', 'Blend content embedding (fresh) with collaborative embedding (accurate).'),
        ],
    },
    'User Embedding Freshness': {
        'Problem': 'User interests change; stale user embeddings cause poor recommendations',
        'Approaches': [
            ('Online user tower', 'Compute user embedding at request time from real-time features. Most common.'),
            ('Session-aware context', 'Concatenate recent session items to user embedding at serving time.'),
            ('Streaming user updates', 'Update user embedding cache every N minutes using Flink.'),
            ('EMA of embeddings', 'Exponential moving average of interaction embeddings.'),
        ],
    },
    'Cold Start Solutions': {
        'Problem': 'New users/items have no interaction history → no embedding',
        'Approaches': [
            ('Content encoder', 'New items: encode title/description/image. No interaction needed.'),
            ('Default embedding', 'Use category-average embedding as starting point.'),
            ('Quick-start features', 'Collect 3-5 interactions → bootstrap embedding. LinkedIn skill prompt.'),
            ('Exploration policy', 'Allocate 5-10% traffic to new items for exploration (UCB/ε-greedy).'),
            ('Transfer learning', 'Initialize new item embeddings from similar known items.'),
        ],
    },
}

for category, details in strategies.items():
    print(f"\n{'='*70}")
    print(f"  {category}")
    print(f"  Problem: {details['Problem']}")
    print(f"{'='*70}")
    for approach, rationale in details['Approaches']:
        print(f"  • {approach}")
        print(f"      → {rationale}")

print("\n" + "="*70)
print("Interview question: How do you handle the cold start problem for new items?")
print("="*70)
print("Strong answer:")
print("  1. Use content encoder (text/image) to generate initial embedding")
print("  2. Allocate 5% exploration traffic to new items (Thompson Sampling)")
print("  3. Once 100+ interactions: retrain item tower, replace content embedding")
print("  4. Monitor new item discovery rate as a key metric")

## Summary

| Concept | Key Point | Interview Signal |
|---------|-----------|------------------|
| Two-tower architecture | Separate encoders for query/user and items; item embs pre-computed | Can draw and explain the serving flow |
| ANN vs exact search | Exact is O(n*d); ANN trades recall for speed (100× speedup) | Knows recall-speed tradeoff |
| HNSW vs IVF-PQ vs ScaNN | HNSW: high recall; IVF-PQ: memory-efficient; ScaNN: CPU throughput | Can justify choice for context |
| Embedding dimension | 64-512 typical; larger = more expressive but more memory | Can estimate memory for 1B items |
| Cold start | Content encoder → bootstrap → collaborative filtering | Has a concrete multi-stage strategy |
| Embedding freshness | Items: batch update; Users: online tower at request time | Knows items and users need different strategies |
| Memory estimation | 1B items × 256-dim × 4 bytes = 1TB — requires distributed or quantization | Can do back-of-envelope |
| Faiss IVF probe tradeoff | More probes = higher recall, more compute | Quantitative answer with trade-off reasoning |